# Prompt Evaluation with LLM-as-Grader

Demonstrates how to:
1. Generate an evaluation dataset with Claude
2. Run each task through a prompt
3. Grade the output using another Claude call (LLM-as-judge pattern)

In [ ]:
%pip install -q anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file"

client = Anthropic()
model = "claude-haiku-4-5"
print("Anthropic client ready.")

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message.content[0].text

## Step 1 — Generate evaluation dataset

In [ ]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompts that produce Python, JSON, or Regex for AWS-related tasks.
Return a JSON array of objects with fields: "task" and "format" (python/json/regex).
Focus on tasks solvable with a single function, single JSON object, or one regex.
Generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(json.dumps(dataset, indent=2))

## Step 2 — Run each task through a prompt and grade the output

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Evaluate the AI-generated solution below.

Task: {test_case["task"]}

Solution:
{output}

Respond with JSON only:
{{"strengths": string[], "weaknesses": string[], "reasoning": string, "score": number (1-10)}}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


def run_prompt(test_case):
    messages = []
    add_user_message(messages, f"Please solve the following task:\n\n{test_case['task']}")
    return chat(messages)


def run_test_case(test_case):
    output = run_prompt(test_case)
    grade = grade_by_model(test_case, output)
    return {
        "output": output,
        "test_case": test_case,
        "score": grade["score"],
        "reasoning": grade["reasoning"],
    }

In [ ]:
from statistics import mean

def run_eval(dataset):
    results = [run_test_case(tc) for tc in dataset]
    print(f"Average score: {mean(r['score'] for r in results):.2f}")
    return results

In [ ]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))